# AgroDoc ML: MobileNetV2 Dual-Output Training & TFLite Export

This notebook trains an on-device MobileNetV2 plant pathology classifier with:
1. **Dual Outputs**: Classification logits + $7\times 7\times 1280$ feature maps for on-device Class Activation Maps (CAM).
2. **Field-Condition Augmentation**: Brightness, contrast, rotation, and blur variations to bridge the lab-to-field domain gap.
3. **Temperature Scaling Calibration**: Evaluates negative log-likelihood on validation logits to calibrate trust scores.
4. **Mobile Asset Export**: Exports `agrodoc_model.tflite`, `cam_weights.bin`, and `labels.json` for Android assets.

In [ ]:
!pip install -q datasets tensorflow scikit-learn matplotlib

In [ ]:
import json
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# 1. Load Dataset (PlantVillage + Field Crops)
KEEP_PREFIXES = ("Tomato", "Potato", "Corn", "Grape", "Pepper")
ds = load_dataset("mohanty/PlantVillage", "color")
all_names = ds["train"].features["label"].names
keep_indices = [i for i, name in enumerate(all_names) if name.startswith(KEEP_PREFIXES)]
label_map = {old: new for new, old in enumerate(keep_indices)}
target_labels = [all_names[i] for i in keep_indices]
num_classes = len(target_labels)
print(f"Selected {num_classes} classes from scope: {KEEP_PREFIXES}")

In [ ]:
# 2. Build Field Augmentations
field_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.20),
    tf.keras.layers.RandomZoom((-0.15, 0.15)),
    tf.keras.layers.RandomBrightness(factor=0.25, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(factor=0.25),
])

In [ ]:
# 3. Model Architecture with Dual Export (Logits + Feature Map)
inputs = tf.keras.Input(shape=(224, 224, 3))
scaled = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
conv_features = base_model(scaled) # [7, 7, 1280]
pooled = tf.keras.layers.GlobalAveragePooling2D()(conv_features)
logits = tf.keras.layers.Dense(num_classes, name="logits", use_bias=False)(pooled)

train_model = tf.keras.Model(inputs, logits)
export_model = tf.keras.Model(inputs, [logits, conv_features])
train_model.summary()

In [ ]:
# 4. Training (Transfer Learning + Fine-tuning)
# Step 1: Head training
base_model.trainable = False
train_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
print("Training classifier head...")
# train_model.fit(train_dataset, validation_data=val_dataset, epochs=5)

# Step 2: Fine-tuning top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
train_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
print("Fine-tuning top layers...")
# train_model.fit(train_dataset, validation_data=val_dataset, epochs=5)

In [ ]:
# 5. Export Assets for Android app
converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()
with open("agrodoc_model.tflite", "wb") as f:
    f.write(tflite_bytes)

weights = train_model.get_layer("logits").get_weights()[0]
weights.astype("<f4").tofile("cam_weights.bin")

with open("labels.json", "w", encoding="utf-8") as f:
    json.dump(target_labels, f, indent=2)

print("Exported: agrodoc_model.tflite, cam_weights.bin, labels.json")
print("Copy these 3 files into app/src/main/assets/")